# Example: Properties of Markov Models and Stationary Distributions
In this example, you will explore the fundamental properties of Markov models by constructing a discrete-time Markov chain, computing its stationary distribution, and validating the theoretical predictions through simulation.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * **Construct and validate a transition matrix:** A transition matrix is valid when its elements are non-negative and each of its rows sums to one. We'll build the transition matrix for a three-state Markov chain and check that every row sums to one.
> * **Compute the stationary distribution:** The stationary distribution is the distribution of states that the chain approaches after many steps. We'll compute it by power iteration and by raising the transition matrix to a large power, and check that every row of that power equals the stationary distribution.
> * **Simulate the Markov chain:** We simulate the chain by sampling each next state from the row of the transition matrix for the current state. We'll compare its state frequencies and transition counts with independent samples from the stationary distribution.

This is an essential foundation for understanding stochastic processes in machine learning and data science. So, let's get started!
___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package and the packages used here.

Let's set up our code environment:

In [1]:
# Load packages and paths -
include(joinpath(@__DIR__, "Include.jl")); # @__DIR__ locates this notebook's setup file

In addition to the Julia standard libraries, we use the [Distributions.jl](https://github.com/JuliaStats/Distributions.jl) package for the categorical distribution, and the [DataFrames.jl](https://dataframes.juliadata.org/stable/) and [PrettyTables.jl](https://github.com/ronisbr/PrettyTables.jl) packages for tables. See the [Julia documentation](https://docs.julialang.org/en/v1/) for the functions and types used here.

### Implementation
[The `stationary_distribution(...)` function](docs/stationary_distribution.md) implements the lecture's power iteration, starting from a uniform distribution. It is defined in [src/Markov.jl](src/Markov.jl) and loaded by the setup file. The iteration stops when the 1-norm of the change in the distribution is at most the tolerance, or when the iteration limit is reached. The result is a $1\times N$ matrix representing a row vector; reaching the limit before the tolerance also produces a warning.

#### Constants
In the simulations below, we'll need some constant values that we set here. In particular, we set a value for the `number_of_states` variable, the `number_of_simulation_steps` variable (the number of steps we take in a Markov chain), and the `number_of_samples` variable:

In [2]:
number_of_states = 3; # how many states do we have?
number_of_simulation_steps = 20000; # number of simulation steps
number_of_samples = 10000; # number of samples
seed = 5800; # reproducible random sequences for the comparison

___

## Task 1: Set up the transition matrix for a three-state Markov model
In this task, we'll set up the transition matrix $\mathbf{P}$ for a three-state [Markov chain model](https://en.wikipedia.org/wiki/Markov_chain). In this example, we have three states $\mathcal{S}=\left\{1,2,3\right\}$, where the probability of moving from state $i$ to state $j$ in the next time step, denoted as $p_{ij}$, is an element of the matrix $\mathbf{P} \in \mathbb{R}^{3\times{3}}$.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-ThreeState-MM-Schematic.svg" width="580" style="max-width:100%; height:auto;" alt="Three circles labeled 1, 2, and 3 in a row. Arrows p12 and p21 connect states 1 and 2, arrows p23 and p32 connect states 2 and 3, and each state has a self-loop labeled p11, p22, and p33. There are no arrows between states 1 and 3."/>
</p>

In [3]:
P = [
    0.05 0.95 0.0 ; # moves for state 1
    0.6 0.2 0.2 ; # moves for state 2
    0.0 0.3 0.7 ; # moves for state 3
];

The rank of the transition matrix $\mathbf{P}$ is the number of linearly independent rows (or columns). Let's compute it now; in Task 2 we'll compare it with the rank of a large power of $\mathbf{P}$.

Our chain is irreducible, because every state can be reached from every other state ($1\rightarrow 2\rightarrow 3$ and $3\rightarrow 2\rightarrow 1$), and aperiodic, because each state can stay where it is for one step ($p_{11}$, $p_{22}$, and $p_{33}$ are all positive). So the chain is ergodic, and it has a unique stationary distribution.

In [4]:
rank(P)

3

### Check: Do the rows of the transition matrix $\mathbf{P}$ sum to `1`?
We know that the rows of the transition matrix $\mathbf{P}$ must sum to `1`. That is, if we are in state $s_{i}\in\mathcal{S}$ at time $t$, then at time $t+1$ we must be in some state $s_{j}\in\mathcal{S}$. 

> __Check:__ Let's verify that the transition matrix $\mathbf{P}$ meets this criterion using [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) by iterating over the rows of the transition matrix $\mathbf{P}$ and checking the sum of each row. We compare with `≈` (approximately equal) because a sum of floating-point numbers can differ from `1` in its last digits. If any row does not meet this criterion, an [`AssertionError`](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) will be thrown.

So what do we see?

In [5]:
for i ∈ 1:number_of_states
    @assert sum(P[i,:]) ≈ 1
end

___

## Task 2: Compute the stationary distribution $\bar{\pi}$
In this task, we'll compute the stationary distribution $\bar{\pi}$ for our example [Markov chain](https://en.wikipedia.org/wiki/Markov_chain).
Let the row vector $\mathbf{\pi}_{n}$ hold the probability of each state at time step $n$, so its elements are non-negative and sum to one. For a finite state space $\mathcal{S}$ and a time-homogeneous transition matrix $\mathbf{P}$, the distribution at time step $n$ is given by:
$$
\begin{equation*}
\mathbf{\pi}_{n} = \mathbf{\pi}_{0}\cdot\left(\mathbf{P}\right)^n
\end{equation*}
$$
where $\mathbf{\pi}_{0}$ is the initial distribution and $\left(\mathbf{P}\right)^{n}$ is the transition matrix raised to the $n$-th power. Each step is a left vector-matrix product: the new distribution is a weighted sum of the rows of $\mathbf{P}$, weighted by the current probabilities.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Matrix-Vector-Left-bA-product.svg" width="580" style="max-width:100%; height:auto;" alt="A row vector with elements a, b, c multiplies a three-by-three matrix whose rows are red, green, and blue. The product equals a times the red row, plus b times the green row, plus c times the blue row, which gives a single row vector."/>
</p>

For an ergodic chain, a unique stationary distribution $\bar{\pi}$ exists, and $\mathbf{P}^{k}$ converges to a rank-one matrix in which each row is the stationary distribution $\bar{\pi}$:
$$
\begin{equation*}
\lim_{k\rightarrow\infty} \mathbf{P}^{k} = \mathbf{1}\,\bar{\pi}
\end{equation*}
$$
where $\mathbf{1}$ is a column vector of all ones, so the product $\mathbf{1}\,\bar{\pi}$ stacks copies of the row vector $\bar{\pi}$.

### Implementation details
We'll compute the stationary distribution $\bar{\pi}$ using [the `stationary_distribution(...)` function](docs/stationary_distribution.md), implementing the pseudocode from the lecture notes.

> __What is going to happen?__ 
>
> We'll iterate until we hit one of two possible conditions:
> * The `counter == maxcount`; at this point, the iteration stops, and the vector $\bar{\pi}$ is returned (but it may not have converged).
> * The iteration also stops when the difference between subsequent estimates of $\bar{\pi}$ is smaller than a specified threshold $\epsilon$. In this case, the estimate has stopped changing, and we accept it as $\bar{\pi}$.

We'll save the stationary distribution in the $\bar{\pi}$ variable.

In [6]:
π̄ = L11aMarkov.stationary_distribution(P, ϵ = 1e-11, maxcount = 10000)

1×3 Matrix{Float64}:
 0.274809  0.435115  0.290076

__So how did this work?__ At each iteration, we update the row distribution using the expression:

$$
\pi_{n+1}=\pi_n\mathbf{P},\qquad \pi_n=\pi_0\mathbf{P}^n.
$$

For an ergodic chain, these distributions approach the stationary distribution. Let's start in state 3 and inspect a few checkpoints. Each update multiplies the current row vector by the transition matrix:

In [7]:
transient_distributions = let

    # Initialize -
    π = reshape([0.0, 0.0, 1.0], 1, number_of_states); # π₀ starts in state 3
    checkpoints = [0, 1, 4, 8, 16, 32, 64]; # time steps to display
    df = DataFrame();

    # Update the distribution one step at a time -
    for n ∈ 0:last(checkpoints)
        if n > 0
            π = π * P;
        end
        if n ∈ checkpoints
            push!(df, (step = n, state_1 = π[1, 1], state_2 = π[1, 2], state_3 = π[1, 3]));
        end
    end

    df; # return the checkpoint distributions
end;
pretty_table(transient_distributions; backend = :text,
    table_format = TextTableFormat(borders = text_table_borders__compact));

 ------- ---------- ---------- ----------
   step    state_1    state_2    state_3 
  Int64    Float64    Float64    Float64 
 ------- ---------- ---------- ----------
      0        0.0        0.0        1.0
      1        0.0        0.3        0.7
      4    0.24255    0.37215     0.3853
      8   0.271305    0.42581   0.302885
     16   0.274793   0.434896   0.290311
     32   0.274809   0.435114   0.290076
     64   0.274809   0.435115   0.290076
 ------- ---------- ---------- ----------


__What do we see?__ The initial distribution puts all probability in state 3. The later rows approach the same stationary distribution computed from the uniform start.

We can also raise the transition matrix to a large power using [the matrix power operator `^`](https://docs.julialang.org/en/v1/base/math/#Base.:^-Tuple{Number,%20Number}) and extract any row of the result:

In [8]:
test = P^(2000) # raise P to a large power

3×3 Matrix{Float64}:
 0.274809  0.435115  0.290076
 0.274809  0.435115  0.290076
 0.274809  0.435115  0.290076

In [9]:
rank(test)

1

### Check: Is the rank condition on the stationary distribution $\bar{\pi}$ correct?
Every row of `test` should agree with the stationary distribution $\bar{\pi}$ to many digits, so `test` should have __numerical rank__ `1`, the rank after ignoring directions smaller than the rounding error. (Strictly, a finite power of our $\mathbf{P}$ keeps full rank; only the limit has rank one.) Let's check whether this condition is true using [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert).
> __Check:__ If we do not meet this criterion, an [`AssertionError`](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) will be thrown, and we should try using a larger power of $\mathbf{P}$. We'll compute the rank using [the `rank(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.rank), which is exported by the [Julia LinearAlgebra package](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/).

What do we see?

In [10]:
@assert rank(test) == 1

___

## Task 3: Compute states using a categorical distribution
In this task, we compute the sequence of states for the three-state [Markov chain model](https://en.wikipedia.org/wiki/Markov_chain). We can obtain the dynamics predicted by the [Markov model](https://en.wikipedia.org/wiki/Markov_model) (i.e., the sequence of states and state transitions) by sampling the transition probability matrix $\mathbf{P}$ directly. 

Now that we are sure that the transition matrix $\mathbf{P}$ is properly formulated, let's populate the `state_probability_dictionary`, which holds the [categorical distribution](https://en.wikipedia.org/wiki/Categorical_distribution) modeling the transition probability for each state $s\in\mathcal{S}$ (i.e., the probability that we transition from state $i$ to state $j$ in the next time step).

In [11]:
state_probability_dictionary = Dict{Int,Categorical}();
for i ∈ 1:number_of_states
    state_probability_dictionary[i] = Categorical(P[i,:])
end
state_probability_dictionary

Dict{Int64, Categorical{P} where P<:Real} with 3 entries:
  2 => Categorical{Float64, Vector{Float64}}(support=Base.OneTo(3), p=[0.6, 0.2…
  3 => Categorical{Float64, Vector{Float64}}(support=Base.OneTo(3), p=[0.0, 0.3…
  1 => Categorical{Float64, Vector{Float64}}(support=Base.OneTo(3), p=[0.05, 0.…

Now, we generate `number_of_simulation_steps` worth of dynamic data by sampling the `state_probability_dictionary`. We store these simulation results in the `simulation_dict` dictionary, where the `key` holds the time index and the `value` is the system's state (i.e., $s_{i}\in\mathcal{S}$).

We start by specifying an initial state `sᵢ = 1`. At each iteration of the loop, we retrieve the [categorical distribution](https://en.wikipedia.org/wiki/Categorical_distribution) corresponding to the current state (i.e., the row in the transition matrix $\mathbf{P}$ corresponding to state $s_{i}$). We generate the state at the next step by drawing a sample using [the `rand(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Base.rand).

In [12]:
simulation_dict = let

    # initialize -
    rng = MersenneTwister(seed); # a fixed seed makes the chain reproducible
    simulation_dict = Dict{Int,Int}();
    sᵢ = 1; # hardcode the start state: we could draw from the stationary distribution
    simulation_dict[1] = sᵢ;

    for i ∈ 2:number_of_simulation_steps

        # get the categorical distribution for sᵢ 
        sᵢ = state_probability_dictionary[sᵢ] |> d -> rand(rng, d);
    
        # capture -
        simulation_dict[i] = sᵢ
    end
    foreach(i -> println("Soln: (t=$(i),s=$(simulation_dict[i]))"), 1:10) # print the first ten steps

    simulation_dict; # return
end;

Soln: (t=1,s=1)
Soln: (t=2,s=1)
Soln: (t=3,s=2)
Soln: (t=4,s=1)
Soln: (t=5,s=2)
Soln: (t=6,s=3)
Soln: (t=7,s=2)
Soln: (t=8,s=1)
Soln: (t=9,s=2)
Soln: (t=10,s=1)


### What happens if we sample $\bar{\pi}$ directly?
The simulation generates each next state from the current state's transition probabilities. To draw independent states with the stationary probabilities, we instead use a [categorical distribution](https://en.wikipedia.org/wiki/Categorical_distribution) with probabilities $\bar{\pi}$. Each draw then has the same distribution regardless of the preceding draw.

Let's save that distribution in `d`:

In [13]:
d = Categorical(π̄[1,:]);

We draw `number_of_samples` independent states and retain the sequence for the comparison. A second fixed seed makes these samples reproducible. Let's inspect the first ten:

In [14]:
stationary_samples = rand(MersenneTwister(seed + 1), d, number_of_samples);
first(stationary_samples, 10)

10-element Vector{Int64}:
 1
 3
 2
 2
 1
 3
 3
 1
 2
 2

### Check: Do the two sequences have similar state frequencies?
For an ergodic chain, the fraction of steps a single trajectory spends in each state approaches the stationary probability. Independent draws from $\bar{\pi}$ approach the same frequencies. Let's compare both sequences with the computed stationary distribution:

In [15]:
let

    # Initialize -
    df = DataFrame();
    chain_states = [simulation_dict[t] for t ∈ 1:number_of_simulation_steps];

    # Count visits to each state in the two sequences -
    for s ∈ 1:number_of_states
        chain_count = count(==(s), chain_states);
        independent_count = count(==(s), stationary_samples);
        push!(df, (state = s, chain_frequency = chain_count / length(chain_states),
            independent_frequency = independent_count / length(stationary_samples),
            stationary_probability = π̄[1, s]));
    end

    # Display the comparison -
    pretty_table(df; backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

 ------- ----------------- ----------------------- ------------------------
  state   chain_frequency   independent_frequency   stationary_probability 
  Int64           Float64                 Float64                  Float64 
 ------- ----------------- ----------------------- ------------------------
      1           0.27255                  0.2797                 0.274809
      2           0.43555                  0.4289                 0.435115
      3            0.2919                  0.2914                 0.290076
 ------- ----------------- ----------------------- ------------------------


__What do we see?__ Both sequences spend similar fractions of their samples in each state. To see whether they have the same dynamics, we also need to count transitions between consecutive states.

### Check: Do the two sequences have the same transitions?
The chain cannot move directly from state 1 to state 3, or from state 3 to state 1, because those entries of $\mathbf{P}$ are zero. Independent stationary draws can produce both pairs. For independent draws, the probability of a consecutive pair is given by:

$$
P(Y_t=i,Y_{t+1}=j)=\bar{\pi}_i\bar{\pi}_j,
$$

where $Y_t$ is the state at position $t$ in the independent sequence, and $\bar{\pi}_i$ is the stationary probability of state $i$. For our matrix, about 8 percent of the independent pairs should be $1\rightarrow3$.

Let's count all nine possible pairs in both sequences. The chain has `number_of_simulation_steps - 1` consecutive pairs, and the independent sequence has `number_of_samples - 1`. Dividing each chain count by its row total estimates the corresponding one-step transition probability. In the table, `iid_count` counts independent pairs, `estimated_Pij` is the row-normalized chain count, and `Pij` is the model probability:

In [16]:
chain_transition_counts, independent_transition_counts = let

    # Initialize -
    chain_counts = zeros(Int, number_of_states, number_of_states);
    independent_counts = zeros(Int, number_of_states, number_of_states);

    # Count consecutive pairs in the chain -
    for t ∈ 1:(number_of_simulation_steps - 1)
        chain_counts[simulation_dict[t], simulation_dict[t + 1]] += 1;
    end

    # Count consecutive pairs in the independent sequence -
    for t ∈ 1:(length(stationary_samples) - 1)
        independent_counts[stationary_samples[t], stationary_samples[t + 1]] += 1;
    end

    chain_counts, independent_counts # return the two count matrices
end;

In [17]:
let

    # Build one row for each possible transition -
    df = DataFrame();
    for i ∈ 1:number_of_states
        row_total = sum(chain_transition_counts[i, :]); # departures from state i
        for j ∈ 1:number_of_states
            estimated_probability = row_total > 0 ? chain_transition_counts[i, j] / row_total : missing;
            push!(df, (from = i, to = j, chain_count = chain_transition_counts[i, j],
                iid_count = independent_transition_counts[i, j],
                estimated_Pij = estimated_probability, Pij = P[i, j]));
        end
    end

    # Display the transition comparison -
    pretty_table(df; backend = :text,
        display_size = (-1, -1),
        table_format = TextTableFormat(borders = text_table_borders__compact));

    # checks -
    @assert sum(chain_transition_counts) == number_of_simulation_steps - 1
    @assert sum(independent_transition_counts) == length(stationary_samples) - 1
    @assert all(chain_transition_counts[P .== 0] .== 0)
end

 ------- ------- ------------- ----------- --------------- ---------
   from      to   chain_count   iid_count   estimated_Pij       Pij 
  Int64   Int64         Int64       Int64         Float64   Float64 
 ------- ------- ------------- ----------- --------------- ---------
      1       1           271         817       0.0497156      0.05
      1       2          5180        1172        0.950284      0.95
      1       3             0         808             0.0       0.0
      2       1          5179        1191        0.594604       0.6
      2       2          1777        1837        0.204018       0.2
      2       3          1754        1260        0.201378       0.2
      3       1             0         788             0.0       0.0
      3       2          1754        1280        0.300445       0.3
      3       3          4084         846        0.699555       0.7
 ------- ------- ------------- ----------- --------------- ---------


__What do we see?__ The chain's counts for $1\rightarrow3$ and $3\rightarrow1$ are zero, while the independent sequence contains both transitions. The estimated transition probabilities from the chain approach the entries of $\mathbf{P}$. Thus, matching state frequencies does not determine the transitions between states. In L11b, we use this same counting and row-normalization idea to estimate character transition probabilities from data. The displayed counts change if we change the seeds, sample sizes, or transition matrix.

___

## Summary
In this example, we explored the fundamental properties of Markov models by constructing a three-state Markov chain, computing its stationary distribution through matrix iteration, and validating theoretical predictions through simulation.

> __Key Takeaways:__
>
> * **Transition matrix structure and properties:** We built the transition matrix for a three-state Markov chain and checked that every row sums to one. Because every state can reach every other state and each state can stay where it is, the chain has a unique stationary distribution.
>
> * **Stationary distribution by iteration:** We computed the stationary distribution with power iteration, stopping when the distribution changed by less than the tolerance between steps. Raising the transition matrix to a large power gave a matrix of numerical rank one whose rows all agree with the stationary distribution.
>
> * **Simulation recovers the stationary distribution:** We simulated 20,000 steps of the chain by sampling each next state from the current state's row of the transition matrix, and the fraction of steps spent in each state was close to the stationary distribution. Independent stationary samples gave similar state frequencies, but the transition counts showed that they also produced pairs forbidden by the chain. Matching state frequencies does not determine a process's dynamics.

Markov models are used to describe random systems in applications ranging from natural language processing to financial modeling and biological sequence analysis.

___